In [ ]:
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- Fetch daily klines from Binance, with error checks ---
def get_daily_klines(symbol="BTCUSDT", limit=500):
    url    = "https://api.binance.com/api/v3/klines"
    params = {"symbol": symbol, "interval": "1d", "limit": limit}
    resp   = requests.get(url, params=params, timeout=10)
    if resp.status_code != 200:
        raise RuntimeError(f"Binance API returned {resp.status_code}: {resp.text}")
    data = resp.json()
    if not isinstance(data, list) or len(data) == 0:
        raise ValueError(f"No data returned for symbol={symbol!r}. Check that the pair and interval exist.")
    cols = [
        "open_time","open","high","low","close","volume",
        "close_time","quote_asset_volume","num_trades",
        "taker_buy_base_volume","taker_buy_quote_volume","ignore"
    ]
    df = pd.DataFrame(data, columns=cols)
    df["open_time"]  = pd.to_datetime(df["open_time"],  unit="ms")
    df["close_time"] = pd.to_datetime(df["close_time"], unit="ms")
    for c in ["open","high","low","close","volume",
              "quote_asset_volume",
              "taker_buy_base_volume","taker_buy_quote_volume"]:
        df[c] = df[c].astype(float)
    # order imbalance = (buys - sells) / total volume
    df["obi"] = (
        df["taker_buy_base_volume"]
        - (df["volume"] - df["taker_buy_base_volume"])
    ) / (df["volume"] + 1e-9)
    df["ret"] = df["close"].pct_change().fillna(0)
    return df

# --- Parameters ---
symbol        = "BTCUSDT"  # <-- changed from BTCUSDC
burn_in_days  = 10         # days to initialize
vol_window    = 10         # days for rolling volatility
T             = 30         # days to run EKF — 個人智慧財產,參數不提供 (placeholder)
c_obi, c_vol  = 1.0, -0.5
dt            = 1.0        # one day

# --- Load data and validate length ---
full_df = get_daily_klines(symbol, limit=burn_in_days + T + vol_window)
required = burn_in_days + T
if len(full_df) < required:
    raise ValueError(f"Need at least {required} days of data for {symbol}, but got {len(full_df)}")

# split into init and run windows
init_df = full_df.iloc[:burn_in_days].reset_index(drop=True)
run_df  = full_df.iloc[burn_in_days:burn_in_days + T].reset_index(drop=True)

# --- Initialize EKF state ---
close_init = init_df["close"].values
t_idx      = np.arange(len(close_init)) * dt
slope, _   = np.polyfit(t_idx, close_init, 1)
x = np.array([close_init[-1], slope])

obi_init = init_df["obi"].values
vol_init = init_df["ret"].rolling(vol_window, min_periods=1).std().values
P = np.diag([
    close_init.var(),
    np.hstack([obi_init, vol_init]).var()
]) * 10

# --- EKF matrices ---
A = np.array([[1, dt],[0, 1]])
B = np.array([[      0,       0],
              [c_obi*dt, c_vol*dt]])
H = np.array([[1, 0]])
Q = np.eye(2) * 0.1
R = np.array([[25]])

# storage for results
dates, obs_price, est_price, est_vel = [], [], [], []

# --- Run EKF day by day ---
for idx, row in run_df.iterrows():
    price, obi = row["close"], row["obi"]
    # rolling vol over past vol_window days
    start = max(0, idx - vol_window)
    rets  = run_df["close"].iloc[start:idx+1].pct_change().dropna()
    vol   = rets.std() if not rets.empty else 0.0

    # predict
    u      = np.array([obi, vol])
    x_pred = A @ x + B @ u
    P_pred = A @ P @ A.T + Q

    # update
    y_pred     = (H @ x_pred)[0]
    innov      = price - y_pred
    S          = H @ P_pred @ H.T + R
    K          = (P_pred @ H.T) @ np.linalg.inv(S)
    x          = x_pred + K.flatten() * innov
    P          = (np.eye(2) - K @ H) @ P_pred

    # record
    dates.append(row["close_time"])
    obs_price.append(price)
    est_price.append(x[0])
    est_vel.append(x[1])

# --- Plot ---
res = pd.DataFrame({
    "date":     dates,
    "observed": obs_price,
    "ekf_est":  est_price,
    "vel_est":  est_vel
})

plt.figure(figsize=(12,5))
plt.plot(res["date"], res["observed"], label="Observed Close", alpha=0.6)
plt.plot(res["date"], res["ekf_est"],  label="EKF Estimate")
plt.title(f"Daily {symbol} Price: Observed vs EKF")
plt.legend(); plt.grid()

plt.figure(figsize=(12,4))
plt.plot(res["date"], res["vel_est"], label="Velocity (Δprice/day)")
plt.axhline(0, linestyle="--", color="gray")
plt.title("EKF Velocity Estimate")
plt.legend(); plt.grid()

plt.show()
